# Unit Testing with Spark

In [0]:
# anatomy of streamin query
kafka_stream_df = spark.readStream.format('kafka').option('kafka.bootstrap.servers', 'kafka-server:9092').option('subscribe', 'topic').load()

kafka_stream_df = kafka_stream_df.selectExpr("cast(value AS STRING) AS json").select(F.from_json('json', schema).alias('data')
                                                                                     
kafka_stream_df.writeStream.format('delta').option('checkpointLocation', '/tmp/checkpoint').trigger(processingTime='10 seconds').start('/tmp/delta')

# batch stream processing
spark.read.table('source_name').withColumn('..').select('..').write.otputMode('append').saveAsTable('sink_table')

# streaming
spark.readStream.table('source_name').withColumn('..').select('..').writeStream.trigger('..').queryName('..').option('checkpointLocation', '/tmp/checkpoint').to_table('sink_table')

# Delta Lake Streaming tunning 
maxFilesPerTrigger - Max files read per micro-batch (def 1k)
maxBytesperTrigger - Soft limit to amount of data read per micro-batch (no default)


In [0]:
# Aggregations, Windows, Watermarks

window_df = (
  events_df
  .withWatermark('event_time', '10 minutes') # wait for late data
  .groupBy(
    F.window('event_time', '10minutes', '5 minutes') # 10 minutes batch window with 5 minutes slide
  ).count()
  .writeStream
  .trigger(processingTime='5 minutes')
)

windowed_df = (
  parsed_df.withWatermark(evenTime='event_column', delayThreshold='90 minutes')
  .groupBy(F.window(timeColumn='event_column', windowDuration='60 minutes'), 'column_name')
  .agg(F.sum('purchase_in_usd').alias('total_revenue'))
)

windowed_df.stream.stop()

for stream in spark.streams.active:
  stream.stop()

# Broadcast small tables

spark.readStream.format('cloudFiles').schema("key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG").option('cloudFiles.format', 'json').load(spark.conf.get('path_to_data')).join(
  F.broadcast(dlt.read('table_name')), # broadcast table
  F.to_date((F.col('timestamp')/ 1000).cast('timestamp')) == F.col("date"), "left"
)

In [0]:
# Stream from multiple source

bpm_schema = "device_id LONG, time TIMESTAMP, hearthrate DOUBLE"

@dlt.table(
    table_properties={"quality": "bronze"}
)
def bpm_bronze():
    return(
        dlt.read_stream('bronze')
        .filter("topic = 'bpm'")
        .select(F.from_json(F.col('value').cast('string'), bpm_schema).alias('v'))
        .select('v.*')
    )

In [0]:
# Data quality enforcement

ruls = { # provide multi rules for table
    'valid_hearthrate': 'hearthrate IS NOT NULL',
    'valid_device_id': 'device_id IS NOT NULL',
    'valid_device_id_range': 'device_id > 11000'
}

@dlt.table(
    table_properties={"quality": "silver"},
)
@dlt.expect_all_or_drop(rules) # provide rules into decorator
def bpm_silver():
    return (
        dlt.read_stream('bpm_bronze')
        .select('*', F.when(F.col('hearthrate') <= 0, 'Negative BPM').otherwise("OK").alias('bpm_check'))
        .withWatermark('time', '30 seconds')
        .dropDuplicates(['device_id', 'time'])
    )

# Create a quarantine table
quarantine_rules = {}
quarantine_rules['invalid_record'] = f{"NOT ({' AND '.join(rules.values())'})"}

@dlt.table
@dlt.expect_all_or_drop(quarantine_rules) # Oposite rules to catch invalid records
def bpm_quarantine():
    return (
        dlt.read_stream('bpm_silver')
    )

In [0]:
from pyspark.tesing.utils import assertDataFrameEqual, assertSchemaEqual

assertDataFrameEqual(actual_df, expected_df)
assertSchemaEqual(actual_df.schema, expected_df.schema)

import pytest, sys

sys.dont_write_bytecode = True

ret_code = pytest.main(['path/to/test/test_module.py', '-v', '-p', 'no:cacheprovider'])
assert ret_code == 0  "Pytest invocation see log for details."